# Data quality checks tutorial: Custom Entity Types

This tutorial shows you how to create and run a DQ check on a set of custom entities.

## Section 1: Create and test a check definition
1. Create the `~Office` custom entity type from [Extending LUSID's data model using custom entities](https://support.lusid.com/docs/extending-lusids-data-model-using-custom-entities).
2. Create property definitions and upsert `~Office` instances.
3. Create a check definition that defines which data to run the checks on.
4. Add a rule to the check definition.
5. Run the check manually.

## Section 2: Set up a DQ check workflow
1. Create exception task definition to handle DQ check results (breaches).
2. Create DQ check task definition.
3. Kick off a task test run.
4. Inspect the results.

## Setup
Build the LUSID and Workflow APIs and create some test data to run a DQ check on

In [ ]:
# import the latest SDK version
#!pip3 install -U finbourne-sdk

import os
import time
from pprint import pprint

import pandas as pd

import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as lm
import finbourne.sdk.services.workflow as lw
import finbourne.sdk.services.workflow.models as lwm

from finbourne.sdk.extensions import SyncApiClientFactory
from finbourne.sdk.exceptions import ApiException

secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

api_factory = SyncApiClientFactory(secrets_path=secrets_path, app_name="LusidJupyterNotebook")

pd.DataFrame(api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().to_dict())

In [ ]:
# Helper functions for compact list construction
def fields(*items):
    return [lwm.TaskFieldDefinition(name=n, type=t, display_name=d) for n, t, d in items]

def states(*names):
    return [lwm.TaskStateDefinition(name=n, display_name=n, description=n) for n in names]

def triggers(*names):
    return [lwm.TransitionTriggerDefinition(name=n, trigger=lwm.TriggerSchema(type="External")) for n in names]

def transitions(*items):
    return [lwm.TaskTransitionDefinition(from_state=f, to_state=t, trigger=tr, **kw) for f, t, tr, *rest in items for kw in [rest[0] if rest else {}]]

def map_from(field):
    return lwm.FieldMapping(map_from=field, set_to=None)

In [ ]:
# Specify a unique scope and code to segregate data in this example
module_scope = "Finbourne-Examples"
module_code = "DQ-Check-ce"
print(f"'{module_scope}\\{module_code}' scope and code created.")

In [ ]:
checkDefinitions_api = api_factory.build(lu.CheckDefinitionsApi)
task_definitions_api = api_factory.build(lw.TaskDefinitionsApi)
tasks_api = api_factory.build(lw.TasksApi)

## Section 1: Create and test a check definition

### Step 1: Create an `~Office` custom entity type

Following [Extending LUSID's data model using custom entities](https://support.lusid.com/docs/extending-lusids-data-model-using-custom-entities): a custom entity type representing a corporate office location, with `address`, `seatingCapacity`, `isHeadOffice`, and `Amenities` fields.

In [ ]:
custom_entity_types_api = api_factory.build(lu.CustomEntityTypesApi)

ce_entity_type_name = f"Office{module_code}"
ce_field_schema = [
    lm.CustomEntityFieldDefinition(name="address", lifetime="Perpetual", type="String", required=False, description="The address of the location"),
    lm.CustomEntityFieldDefinition(name="seatingCapacity", lifetime="TimeVariant", type="Decimal", required=False, description="The seating capacity of the location"),
    lm.CustomEntityFieldDefinition(name="isHeadOffice", lifetime="TimeVariant", type="Boolean", required=True, description="Whether or not the location is a head office"),
    lm.CustomEntityFieldDefinition(name="Amenities", lifetime="TimeVariant", type="String", collection_type="Array", required=False, description="A list of facilities for staff"),
]

try:
    create_ce_type_response = custom_entity_types_api.create_custom_entity_type(
        lm.CreateCustomEntityTypeRequest(
            entity_type_name=ce_entity_type_name,
            display_name="Office location",
            description="An office or branch location",
            field_schema=ce_field_schema
        )
    )
    print(f"Custom Entity Type {create_ce_type_response.entity_type} has been created")
except ApiException as e:
    if e.status == 400 and "CustomEntityDefinitionAlreadyExists" in str(e.body):
        print(f"Custom Entity Type '{ce_entity_type_name}' already exists, skipping creation.")
    else:
        raise

### Step 2: Create property definitions and upsert `~Office` instances

#### Create the identifier and RefreshData properties

In [ ]:
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)

ce_scope = f"{module_scope}{module_code}"

ce_identifier_property_definition = lm.CreatePropertyDefinitionRequest(
    domain="CustomEntity",
    scope=ce_scope,
    code="OfficeId",
    display_name="Office ID",
    data_type_id=lm.ResourceId(scope="system", code="string"),
    life_time="Perpetual",
    constraint_style="Identifier",
    property_description="Identifier property used to identify custom entities that are office locations"
)

try:
    property_definitions_api.create_property_definition(
        create_property_definition_request=ce_identifier_property_definition
    )
    print(f"Property {ce_identifier_property_definition.domain}/{ce_identifier_property_definition.scope}/{ce_identifier_property_definition.code} created.")
except ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print(f"Property {ce_identifier_property_definition.domain}/{ce_identifier_property_definition.scope}/{ce_identifier_property_definition.code} already exists, skipping creation.")
    else:
        raise

In [ ]:
ce_refresh_data_prop_definition = lm.CreatePropertyDefinitionRequest(
    domain="CustomEntity",
    scope=ce_scope,
    code="RefreshData",
    display_name="RefreshData",
    property_description="Tags data that should be included in integration runs",
    data_type_id=lm.ResourceId(scope="system", code="string"),
    life_time="Perpetual",
    constraint_style="Property"
)

try:
    property_definitions_api.create_property_definition(
        create_property_definition_request=ce_refresh_data_prop_definition
    )
    print(f"Property {ce_refresh_data_prop_definition.domain}/{ce_refresh_data_prop_definition.scope}/{ce_refresh_data_prop_definition.code} created.")
except ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print("Property 'RefreshData' already exists, skipping creation.")
    else:
        raise

#### Upsert `~Office` instances

Three offices, giving a mix of results for our rule later:

| Office | isHeadOffice | seatingCapacity |
|---|---|---|
| One Carter Lane (London) | True | 150 |
| New York Office | True | *(not set)* |
| Manchester Office | False | *(not set, but excluded by the filter anyway)* |

In [ ]:
custom_entities_api = api_factory.build(lu.CustomEntitiesApi)

def upsert_office(identifier_value, display_name, description, address, is_head_office, seating_capacity=None):
    office_fields = [
        lm.CustomEntityField(name="address", value=address),
        lm.CustomEntityField(name="isHeadOffice", value=is_head_office),
    ]
    if seating_capacity is not None:
        office_fields.append(lm.CustomEntityField(name="seatingCapacity", value=seating_capacity))

    ce_refresh_data_property_key = f"CustomEntity/{ce_scope}/RefreshData"

    try:
        response = custom_entities_api.upsert_custom_entity(
            f"~Office{module_code}",
            lm.CustomEntityRequest(
                display_name=display_name,
                description=description,
                identifiers=[
                    lm.CustomEntityId(
                        identifier_scope=ce_scope,
                        identifier_type="OfficeId",
                        identifier_value=identifier_value
                    )
                ],
                fields=office_fields,
                properties={
                    ce_refresh_data_property_key: lm.ModelProperty(
                        key=ce_refresh_data_property_key,
                        value=lm.PropertyValue(label_value="True")
                    )
                }
            )
        )
        print(f"Office '{display_name}' upserted with identifier '{identifier_value}'.")
        return response
    except ApiException as e:
        print(f"Error upserting office '{display_name}': {e}")

upsert_office("London", "One Carter Lane", "FINBOURNE office and regional headquarters", "One Carter Lane, London, EC4V 5ER", True, 150)
upsert_office("NewYork", "New York Office", "A head office missing seating capacity data", "123 Example Ave, New York, NY", True)
upsert_office("Manchester", "Manchester Office", "A branch office, not a head office", "1 Example Street, Manchester", False)

### Step 3: Create a check definition that contains empty rulesets


In [ ]:
custom_entity_rule_set_key = "office-headquarters-checks"
ce_rule_sets = [lm.UpdateCheckDefinitionRuleSet(
    rule_set_key = custom_entity_rule_set_key,
    display_name = "Office headquarters checks ruleset",
    description = "A set of rules to apply to Office entities that are head offices.",
    rule_set_filter = "fields[isHeadOffice] eq true"
)]
try:
    create_ce_check_definition_request = lm.CreateCheckDefinitionRequest(
        id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-custom-entity-properties"
        ),
        display_name = "Office check",
        description = "A check definition to validate head offices have seating capacity populated",
        dataset_schema = lm.CheckDefinitionDatasetSchema(
            type = "LusidEntity",
            entity_type = f"~Office{module_code}"
        ),
        rule_sets = ce_rule_sets
    )

    create_ce_check_definition_response = checkDefinitions_api.create_check_definition(create_check_definition_request=create_ce_check_definition_request)
    pprint(create_ce_check_definition_response)
except ApiException as e:
    if e.status == 400 and "EntityWithIdAlreadyExists" in str(e.body):
        print(f"Check definition '{module_code}-custom-entity-properties' already exists in scope '{module_scope}', skipping creation.")
    else:
        raise

### Step 4: Add a rule to the check definition

In [ ]:
# We want to create the following rule:
# "Check that head offices have seatingCapacity populated"
ce_check_definition_rule = lm.CheckDefinitionRule(
    rule_key = "seating-capacity-populated",
    display_name = "Seating capacity populated check",
    description = "Checks that head offices have the seatingCapacity field populated",
    rule_formula = "fields[seatingCapacity] exists",
    severity = 1
)
try:
    upsert_ce_data_quality_rule = [lm.UpsertDataQualityRule(
        rule_set_key=custom_entity_rule_set_key,
        rule=ce_check_definition_rule
    )]

    upsert_ce_rule_response = checkDefinitions_api.upsert_rules(scope=module_scope, code=f"{module_code}-custom-entity-properties", upsert_data_quality_rule=upsert_ce_data_quality_rule)

    print(f"Successfully upserted rule '{ce_check_definition_rule.rule_key}' to ruleset '{custom_entity_rule_set_key}'.")
    pprint(upsert_ce_rule_response)
except ApiException as e:
    print(f"Error creating ruleset: {e}")

### Step 5: Run the check on some data

Based on the three offices created earlier, we'd expect **1 breach**: New York Office is a head office missing `seatingCapacity`. One Carter Lane is a head office with `seatingCapacity` populated, so it should pass. Manchester Office is excluded entirely by the `ruleSetFilter`, since it isn't a head office.

In [ ]:
try:
    run_ce_check_request = lm.RunCheckRequest(
        lusid_entity_dataset = lm.LusidEntityDataset(
            selector_attribute = f"Properties[CustomEntity/{ce_scope}/RefreshData]",
            selector_value = "True",
            return_identifier_key = f"CustomEntity/{ce_scope}/OfficeId"
        ),
        limit_individual_breaches_per_rule = 100
    )

    run_ce_check_response = checkDefinitions_api.run_check_definition(scope=module_scope, code=f"{module_code}-custom-entity-properties", run_check_request=run_ce_check_request)
    ce_results = run_ce_check_response.data_quality_check_results
    ce_total = len(ce_results)
    print(f"Check complete: {ce_total} results (breaches)")
except ApiException as e:
    print(f"Error running check: {e}")

## Section 2: Set up a DQ check workflow
### Step 1: Create exception task definition


In [ ]:
custom_entity_exception_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-custom-entity-exception"),
    display_name="DQ Custom Entity Exception",
    description="An exception returned by a data quality check.",
    states=states("Pending", "InProgress", "Resolved"),
    field_schema=fields(
        ("checkDefinitionScope",       "String",   "CheckDef Scope"),
        ("checkDefinitionCode",        "String",   "CheckDef Code"),
        ("checkDefinitionDisplayName", "String",   "CheckDef Name"),
        ("checkRunAsAt",               "DateTime", "Run AsAt"),
        ("resultType",                 "String",   "Result Type"),
        ("rulesetKey",                 "String",   "Ruleset Key"),
        ("rulesetDisplayName",         "String",   "Ruleset Name"),
        ("ruleKey",                    "String",   "Rule Key"),
        ("ruleDisplayName",            "String",   "Rule Name"),
        ("ruleDescription",            "String",   "Rule Description"),
        ("ruleFormula",                "String",   "Rule Formula"),
        ("severity",                   "Decimal",  "Severity"),
        ("resultId",                   "String",   "Result Tracking ID"),
        ("entityType",                 "String",   "Entity Type"),
        ("customEntityAsAt",           "DateTime", "Custom Entity As At"),
        ("customEntityEffectiveAt",    "DateTime", "Custom Entity Effective At"),
        ("identifierType",             "String",   "Identifier Type"),
        ("identifierValue",            "String",   "Identifier Value"),
        ("customEntityName",           "String",   "Custom Entity Name"),
        ("entityUniqueId",             "String",   "Entity Unique ID"),
        ("countRuleBreaches",          "Decimal",  "Number of Breaches"),
        ("errorDetail",                "String",   "Error Message"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "resolve"),
    actions=[
        lwm.ActionDefinition(
            name="resolve-parent",
            action_details=lwm.ActionDetails(
                lwm.TriggerParentTaskAction(type="TriggerParentTask", trigger="resolve")
            )
        )
    ],
    transitions=transitions(
        ("Pending",    "InProgress", "start"),
        ("InProgress", "Resolved",   "resolve", {"action": "resolve-parent"}),
    )
)
try:
    custom_entity_exception_response = task_definitions_api.create_task_definition(
        create_task_definition_request=custom_entity_exception_task_def
    )
    print(f"Task definition created successfully. Scope: {custom_entity_exception_response.id.scope}, Code: {custom_entity_exception_response.id.code}")
except ApiException as e:
    print(f"Error creating exception task definition: {e}")

### Step 2: Create DQ check task definition


In [ ]:
custom_entity_child_task_fields = {k: map_from(v) for k, v in [
    ("checkDefinitionScope",       "checkDefinitionScope"),
    ("checkDefinitionCode",        "checkDefinitionCode"),
    ("checkDefinitionDisplayName", "checkDefinitionDisplayName"),
    ("checkRunAsAt",               "checkRunAsAt"),
    ("resultType",                 "resultType"),
    ("rulesetKey",                 "rulesetKey"),
    ("rulesetDisplayName",         "rulesetDisplayName"),
    ("ruleKey",                    "ruleKey"),
    ("ruleDisplayName",            "ruleDisplayName"),
    ("ruleDescription",            "ruleDescription"),
    ("ruleFormula",                "ruleFormula"),
    ("resultId",                   "resultId"),
    ("entityType",                 "lusidEntityType"),
    ("customEntityAsAt",           "lusidEntityAsAt"),
    ("customEntityEffectiveAt",    "lusidEntityEffectiveAt"),
    ("identifierType",             "lusidEntityIdentifierKey"),
    ("identifierValue",            "lusidEntityIdentifierValue"),
    ("customEntityName",           "lusidEntityDisplayName"),
    ("entityUniqueId",             "lusidEntityUniqueId"),
    ("severity",                   "severity"),
    ("countRuleBreaches",          "countRuleBreaches"),
    ("errorDetail",                "errorDetail"),
]}

custom_entity_dq_check_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckCustomEntities"),
    display_name="DQ Check Custom Entities",
    description="Runs data quality checks for custom entities.",
    states=states("Pending", "InProgress", "ExceptionManagement", "Complete", "Error"),
    field_schema=fields(
        ("checkDefinitionScope", "String",  "CD Scope"),
        ("checkDefinitionCode",  "String",  "CD Code"),
        ("selectorAttribute",    "String",  "Selector Attribute"),
        ("selectorValue",        "String",  "Selector Value"),
        ("preferredIdentifier",  "String",  "Preferred Identifier"),
        ("ruleBreachLimit",      "Decimal", "Rule Breach Limit"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "exceptions", "no_exceptions", "resolve", "error"),
    actions=[
        lwm.ActionDefinition(
            name="run-checks",
            action_details=lwm.ActionDetails(
                lwm.RunWorkerAction(
                    type="RunWorker",
                    worker_id=lwm.ResourceId(scope="default", code="LusidEntityDataQuality"),
                    worker_parameters={
                        "checkDefinitionScope":           map_from("checkDefinitionScope"),
                        "checkDefinitionCode":            map_from("checkDefinitionCode"),
                        "selectorAttribute":              map_from("selectorAttribute"),
                        "selectorValue":                  map_from("selectorValue"),
                        "returnIdentifierKey":            map_from("preferredIdentifier"),
                        "limitIndividualBreachesPerRule": map_from("ruleBreachLimit"),
                    },
                    worker_status_triggers=lwm.WorkerStatusTriggers(
                        started=None,
                        completed_with_results="exceptions",
                        completed_no_results="no_exceptions",
                        failed_to_start="error",
                        failed_to_complete="error"
                    ),
                    child_task_configurations=[
                        lwm.ResultantChildTaskConfiguration(
                            task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-custom-entity-exception"),
                            map_stacking_key_from="resultId",
                            child_task_fields=custom_entity_child_task_fields,
                            result_matching_pattern=None,
                            initial_trigger=None
                        )
                    ]
                )
            )
        )
    ],
    transitions=transitions(
        ("Pending",             "InProgress",          "start",         {"action": "run-checks"}),
        ("InProgress",          "Complete",            "no_exceptions"),
        ("InProgress",          "ExceptionManagement", "exceptions"),
        ("ExceptionManagement", "Complete",            "resolve",       {"guard": "childTasks all (state eq 'Resolved')"}),
        ("InProgress",          "Error",               "error"),
    )
)

try:
    custom_entity_dq_check_response = task_definitions_api.create_task_definition(
        create_task_definition_request=custom_entity_dq_check_task_def
    )
    print(f"Task definition created successfully. Scope: {custom_entity_dq_check_response.id.scope}, Code: {custom_entity_dq_check_response.id.code}")
except ApiException as e:
    print(f"Error creating DQ check task definition: {e}")

### Step 3: Kick off a task test run

In [ ]:
create_custom_entity_task_request = lwm.CreateTaskRequest(
    task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckCustomEntities"),
    correlation_ids=[],
    fields=[
        lwm.TaskInstanceField(
            name="checkDefinitionScope",
            value=module_scope
        ),
        lwm.TaskInstanceField(
            name="checkDefinitionCode",
            value=f"{module_code}-custom-entity-properties"
        ),
        lwm.TaskInstanceField(
            name="selectorAttribute",
            value=f"Properties[CustomEntity/{ce_scope}/RefreshData]"
        ),
        lwm.TaskInstanceField(
            name="selectorValue",
            value="True"
        ),
        lwm.TaskInstanceField(
            name="preferredIdentifier",
            value=f"CustomEntity/{ce_scope}/OfficeId"
        ),
        lwm.TaskInstanceField(
            name="ruleBreachLimit",
            value="100"
        ),
    ]
)

try:
    create_custom_entity_task_response = tasks_api.create_task(
        create_task_request=create_custom_entity_task_request,
        trigger="start"
    )
    custom_entity_task_id = create_custom_entity_task_response.id
    print(f"Task created successfully. ID: {custom_entity_task_id}")
except ApiException as e:
    print(f"Error creating task: {e}")

### Step 4: Inspect the results

Polls for child tasks every 10 seconds, up to a maximum of 2 minutes, stopping as soon as results are found.

In [ ]:
def display_custom_entity_exception_tasks(tasks):
    rows = []
    for task in tasks:
        field_dict = {f.name: f.value for f in task.fields}
        rows.append({
            "Task ID":            task.id,
            "State":              task.state,
            "Custom Entity Name": field_dict.get("customEntityName"),
            "Identifier":         f"{field_dict.get('identifierType')}: {field_dict.get('identifierValue')}",
            "Rule":               field_dict.get("ruleDisplayName"),
            "Rule Formula":       field_dict.get("ruleFormula"),
            "Result Type":        field_dict.get("resultType"),
            "Severity":           field_dict.get("severity"),
            "Ruleset":            field_dict.get("rulesetDisplayName"),
            "Check Run At":       field_dict.get("checkRunAsAt"),
        })
    display(pd.DataFrame(rows))

max_attempts = 12
wait_seconds = 10
all_custom_entity_exception_tasks = []

for attempt in range(max_attempts):
    print(f"Checking for child tasks (attempt {attempt + 1}/{max_attempts})...")
    try:
        response = tasks_api.list_tasks(
            filter=f"ultimateParentTask.id eq '{custom_entity_task_id}'"
        )
        found = [t for t in response.values if t.parent_task is not None]

        while response.next_page:
            response = tasks_api.list_tasks(
                filter=f"ultimateParentTask.id eq '{custom_entity_task_id}'",
                page=response.next_page
            )
            found.extend([t for t in response.values if t.parent_task is not None])

        if found:
            all_custom_entity_exception_tasks = found
            print(f"Found {len(all_custom_entity_exception_tasks)} exception tasks.")
            break
    except ApiException as e:
        print(f"Error retrieving child tasks: {e}")
        break

    time.sleep(wait_seconds)
else:
    print("No exception tasks found after maximum wait time.")

if all_custom_entity_exception_tasks:
    display_custom_entity_exception_tasks(all_custom_entity_exception_tasks)

## Next steps

You can then resolve breaks and manage your tasks via the LUSID web app. [Read more.](https://support.lusid.com/docs/how-do-i-set-up-a-data-quality-check-workflow)


## Teardown


In [ ]:
# Task definitions (delete before the check definition/custom entities they reference)
try:
    task_definitions_api.delete_task_definition(scope=module_scope, code=f"{module_code}-CheckCustomEntities")
    print(f"Deleted task definition '{module_code}-CheckCustomEntities'.")
except ApiException as e:
    if e.status == 404:
        print(f"Task definition '{module_code}-CheckCustomEntities' did not exist, skipping.")
    else:
        print(f"Error deleting task definition '{module_code}-CheckCustomEntities': {e}")

try:
    task_definitions_api.delete_task_definition(scope=module_scope, code=f"{module_code}-custom-entity-exception")
    print(f"Deleted task definition '{module_code}-custom-entity-exception'.")
except ApiException as e:
    if e.status == 404:
        print(f"Task definition '{module_code}-custom-entity-exception' did not exist, skipping.")
    else:
        print(f"Error deleting task definition '{module_code}-custom-entity-exception': {e}")

# Check definition
try:
    checkDefinitions_api.delete_check_definition(scope=module_scope, code=f"{module_code}-custom-entity-properties")
    print(f"Deleted check definition '{module_code}-custom-entity-properties'.")
except ApiException as e:
    if e.status == 404:
        print(f"Check definition '{module_code}-custom-entity-properties' did not exist, skipping.")
    else:
        print(f"Error deleting check definition '{module_code}-custom-entity-properties': {e}")

# Custom entity instances
try:
    custom_entities_api.delete_custom_entity(entity_type=f"~Office{module_code}", identifier_type="OfficeId", identifier_value="London", identifier_scope=ce_scope)
    print("Deleted custom entity 'London'.")
except ApiException as e:
    if e.status == 404:
        print("Custom entity 'London' did not exist, skipping.")
    else:
        print(f"Error deleting custom entity 'London': {e}")

try:
    custom_entities_api.delete_custom_entity(entity_type=f"~Office{module_code}", identifier_type="OfficeId", identifier_value="NewYork", identifier_scope=ce_scope)
    print("Deleted custom entity 'NewYork'.")
except ApiException as e:
    if e.status == 404:
        print("Custom entity 'NewYork' did not exist, skipping.")
    else:
        print(f"Error deleting custom entity 'NewYork': {e}")

try:
    custom_entities_api.delete_custom_entity(entity_type=f"~Office{module_code}", identifier_type="OfficeId", identifier_value="Manchester", identifier_scope=ce_scope)
    print("Deleted custom entity 'Manchester'.")
except ApiException as e:
    if e.status == 404:
        print("Custom entity 'Manchester' did not exist, skipping.")
    else:
        print(f"Error deleting custom entity 'Manchester': {e}")

# Property definitions
try:
    property_definitions_api.delete_property_definition(domain="CustomEntity", scope=ce_scope, code="OfficeId")
    print(f"Deleted property definition 'CustomEntity/{ce_scope}/OfficeId'.")
except ApiException as e:
    if e.status == 404:
        print(f"Property definition 'CustomEntity/{ce_scope}/OfficeId' did not exist, skipping.")
    else:
        print(f"Error deleting property definition 'CustomEntity/{ce_scope}/OfficeId': {e}")

try:
    property_definitions_api.delete_property_definition(domain="CustomEntity", scope=ce_scope, code="RefreshData")
    print(f"Deleted property definition 'CustomEntity/{ce_scope}/RefreshData'.")
except ApiException as e:
    if e.status == 404:
        print(f"Property definition 'CustomEntity/{ce_scope}/RefreshData' did not exist, skipping.")
    else:
        print(f"Error deleting property definition 'CustomEntity/{ce_scope}/RefreshData': {e}")

# Custom entity type (delete last, since instances/properties reference it)
try:
    custom_entity_types_api.delete_custom_entity_type(entity_type=f"~Office{module_code}")
    print(f"Deleted custom entity type 'Office{module_code}'.")
except ApiException as e:
    if e.status == 404:
        print(f"Custom entity type 'Office{module_code}' did not exist, skipping.")
    else:
        print(f"Error deleting custom entity type 'Office{module_code}': {e}")